# Chapter 5: Requirements Elicitation with LLMs

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Smart Requirements Extractor** that takes unstructured inputs (meeting notes, emails, interview transcripts) and extracts structured requirements with traceability.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Understanding of requirements engineering basics


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Extracting Requirements from Meeting Notes

Real-world requirements often come from messy, unstructured sources. Let's extract structured requirements from stakeholder meeting notes.


In [ ]:
# Simulated meeting transcript
meeting_notes = """
Project: Customer Portal Redesign
Date: 2024-11-15
Attendees: Sarah (Product Owner), Mike (Tech Lead), Lisa (UX), Tom (BA)

Sarah: We need the new portal to load much faster than what we have now. 
Customers are complaining about the dashboard taking forever.

Mike: We can do lazy loading. But we also need to think about the search -- 
right now it's doing full table scans. We should add elasticsearch.

Lisa: From the user research, customers want a way to customize their 
dashboard. They want to drag and drop widgets. Also, the mobile experience 
is terrible -- we need responsive design.

Sarah: Oh, and we need SSO integration with Okta. Legal says we must also 
add an audit trail for all data access. That's non-negotiable for SOC2.

Tom: What about data export? I've heard that request multiple times.

Sarah: Yes, CSV and PDF export for reports. And Mike, can we add real-time 
notifications? Push notifications for order status changes.

Mike: We'll need WebSocket support for that. Also, we should plan for 
multi-tenancy since the enterprise clients want isolated environments.
"""

# Extract requirements from meeting notes
extraction_prompt = f"""Extract all requirements from these meeting notes. For each requirement:

1. Assign an ID (REQ-001, REQ-002, etc.)
2. Write a clear, testable requirement statement
3. Classify as: Functional, Non-Functional (Performance/Security/Usability/Scalability)
4. Identify the source (who mentioned it)
5. Assign priority: Must-Have, Should-Have, Nice-to-Have
6. Identify any dependencies on other extracted requirements

Meeting Notes:
{meeting_notes}

Return as a JSON array. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': extraction_prompt}],
    temperature=0.2
)
requirements = json.loads(response.choices[0].message.content)
print(f'Extracted {len(requirements)} requirements')
for req in requirements[:5]:
    print(f"\n{req['id']}: {req.get('statement', req.get('description', ''))[:80]}")
    print(f"  Type: {req.get('classification', req.get('type', ''))} | Source: {req.get('source', '')} | Priority: {req.get('priority', '')}")

## 2. Interactive Requirements Clarification

Use the LLM to identify gaps and generate clarification questions.


In [ ]:
# Generate clarification questions
clarify_prompt = f"""As a senior business analyst, review these extracted requirements 
and identify gaps or ambiguities that need clarification.

Requirements:
{json.dumps(requirements, indent=2)}

For each gap, provide:
1. Which requirement(s) it relates to
2. The specific question to ask the stakeholder
3. Why this clarification is important
4. Suggested default if stakeholder is unavailable

Return as a JSON array with fields: related_reqs, question, importance, suggested_default.
Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': clarify_prompt}],
    temperature=0.3
)
questions = json.loads(response.choices[0].message.content)
print(f'Generated {len(questions)} clarification questions:\n')
for i, q in enumerate(questions, 1):
    print(f"{i}. [{', '.join(q['related_reqs']) if isinstance(q['related_reqs'], list) else q['related_reqs']}]")
    print(f"   Q: {q['question']}")
    print(f"   Why: {q['importance']}")
    print()

In [ ]:
# Generate a requirements traceability matrix
trace_prompt = f"""Create a requirements traceability matrix from these requirements.

Requirements: {json.dumps(requirements, indent=2)}

Group requirements by:
1. Feature area (e.g., Dashboard, Search, Authentication, Notifications, etc.)
2. Show dependencies between requirements
3. Suggest an implementation order based on dependencies and priority
4. Identify any conflicting requirements

Format as a readable text report with sections."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': trace_prompt}],
    temperature=0.3
)
print(response.choices[0].message.content)

## Exercises
1. Feed in a real email thread or Slack conversation and extract requirements from it.
2. Add a step that converts extracted requirements into Jira-ready JSON format.
3. Build a multi-turn conversation where the LLM acts as an interviewer, asking elicitation questions.
4. Create a requirements conflict detector that checks a new requirement against existing ones.
